In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/processed/task2_ielts_dataset_criteria.csv')

print("Dataset shape:", df.shape)

df.head()

In [ ]:
print("Missing values:\n")
print(df.isnull().sum())

print("\nOverall missing:", df["Overall_Clean"].isnull().sum())

In [ ]:
conflict_q = (
    df.groupby("Essay")["Question"]
    .nunique()
)

bad_q = conflict_q[conflict_q > 1].index

print("Essays with multiple questions:", len(bad_q))

df = df[~df["Essay"].isin(bad_q)]

print("After removing multi-question:", df.shape)

In [ ]:
q1 = df["length"].quantile(0.25)
q3 = df["length"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("IQR bounds:", lower, upper)

df = df[(df["length"] >= lower) & (df["length"] <= upper)]

print("After IQR filter:", df.shape)

In [ ]:
q1 = df["Overall_Clean"].quantile(0.25)
q3 = df["Overall_Clean"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("IQR bounds:", lower, upper)

df = df[(df["Overall_Clean"] >= lower) & (df["Overall_Clean"] <= upper)]

print("After IQR filter:", df.shape)

In [ ]:
essay_multi_score = (
    df.groupby("Essay")["Overall_Clean"]
    .nunique()
    .value_counts()
)

print("Essay -> number of unique Overall scores:\n")
print(essay_multi_score)

print("Unique Overall scores:")
print(sorted(df["Overall_Clean"].unique()))

In [ ]:
bins = np.arange(1, 9.5, 0.5)

hist = plt.hist(
    df["Overall_Clean"],
    bins=bins,
    edgecolor='black',
    alpha=0.7
)

plt.title("Distribution of IELTS Scores")
plt.xlabel("IELTS Scores")
plt.ylabel("Frequency")

heights = hist[0]
edges = hist[1]

for i in range(len(heights)):
    x = (edges[i] + edges[i+1]) / 2
    y = heights[i]

    if y > 0:
        plt.text(x, y, str(int(y)), ha='center', va='bottom')

plt.xticks(np.arange(1, 9.5, 0.5))

plt.show()

In [ ]:
score_counts = df["Overall_Clean"].value_counts().sort_index()

score_counts

score_counts.plot(kind="bar")

plt.title("Score Class Distribution")
plt.xlabel("Band Score")
plt.ylabel("Number of Essays")

plt.show()

print("Skewness:", df["Overall_Clean"].skew())
print("Kurtosis:", df["Overall_Clean"].kurtosis())

Raw dataset may contain missing values in criteria columns,
which will be handled during preprocessing.

In [ ]:
cumulative_freq = (
    df
    .groupby("Overall_Clean")
    .agg(Count=("Overall_Clean", "count"))
    .sort_index()
)

cumulative_freq["Cumulative_Frequency"] = cumulative_freq["Count"].cumsum()

total_count = cumulative_freq["Count"].sum()

cumulative_freq["Percentage"] = round(
    (cumulative_freq["Cumulative_Frequency"] / total_count) * 100,
    3
)

cumulative_freq

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    cumulative_freq.index,
    cumulative_freq["Percentage"],
    marker='o'
)

plt.title("Cumulative Percentage of IELTS Scores")
plt.xlabel("Band Score")
plt.ylabel("Percentage (%)")

plt.xticks(np.arange(1, 9.5, 0.5))

plt.grid()

plt.show()

In [ ]:
df["length"] = df["Essay"].str.split().str.len()

df["length"].describe()

plt.hist(
    df["length"],
    bins=50,
    edgecolor="black"
)

plt.title("Essay Length Distribution")
plt.xlabel("Word Count")
plt.ylabel("Frequency")

plt.show()

In [ ]:
length_by_score = (
    df
    .groupby("Overall_Clean")
    .agg(mean_length=("length","mean"))
    .reset_index()
)

length_by_score

In [ ]:
plt.plot(
    length_by_score["Overall_Clean"],
    length_by_score["mean_length"],
    marker='o'
)

plt.title("Average Essay Length by Band Score")
plt.xlabel("Band Score")
plt.ylabel("Average Word Count")

plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(
    df["Overall_Clean"],
    df["length"],
    alpha=0.5
)

a, b = np.polyfit(df["Overall_Clean"], df["length"], 1)

x_line = np.linspace(
    df["Overall_Clean"].min(),
    df["Overall_Clean"].max(),
    100
)

y_line = a * x_line + b

plt.plot(
    x_line,
    y_line,
    color='red',
    label=f'y = {a:.2f}x + {b:.2f}'
)

plt.title("Essay Length vs Overall Score")
plt.xlabel("Overall Score")
plt.ylabel("Essay Length")

plt.xticks(np.arange(1, 9.5, 0.5))

plt.grid()
plt.legend()

plt.show()

In [ ]:
score_variance = (
    df
    .groupby("Overall_Clean")
    .agg(
        mean_length=("length","mean"),
        std_length=("length","std"),
        count=("length","count")
    )
)

score_variance

In [ ]:
plt.errorbar(
    score_variance.index,
    score_variance["mean_length"],
    yerr=score_variance["std_length"],
    fmt='o'
)

plt.title("Score Variance Analysis")
plt.xlabel("Band Score")
plt.ylabel("Essay Length (Mean ± Std)")

plt.grid()
plt.show()

In [ ]:
print("Very short essays:")
df[df["length"] < 100][["Essay","Overall_Clean"]].head()

print("\nVery long essays:")
df[df["length"] > 300][["Essay","Overall_Clean"]].head()

In [ ]:
bins = np.arange(1, 9.5, 0.5)

plt.hist(df["Overall_Clean"], bins=bins, edgecolor='black')
plt.title("Overall Score Distribution")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
criteria = [
    "Task_Response",
    "Coherence_Cohesion",
    "Lexical_Resource",
    "Range_Accuracy"
]

for col in criteria:
    plt.hist(df[col], bins=np.arange(1,9.5,0.5), edgecolor="black")
    plt.title(f"{col} Distribution")
    plt.xlabel("Score")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
corr = df[criteria + ["Overall_Clean"]].corr()

corr

In [ ]:
plt.imshow(corr, cmap='coolwarm', interpolation='none')
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha='center', va='center',
            color='black'
        )

plt.title("Correlation Matrix")
plt.show()

In [ ]:
def round_band(x):
    if pd.isna(x):
        return np.nan

    integer = int(x)
    decimal = x - integer

    if decimal < 0.25:
        return integer
    elif decimal <= 0.5:
        return integer + 0.5
    elif decimal < 0.75:
        return integer + 0.5
    else:
        return integer + 1

In [ ]:
print("Number of unique questions:", df["Question"].nunique())

question_counts = df["Question"].value_counts()

question_counts.head()

In [ ]:
question_counts.plot(kind="hist", bins=50)

plt.title("Number of Essays per Question")
plt.xlabel("Count")
plt.ylabel("Frequency")

plt.show()

In [ ]:
import re


In [ ]:
def basic_clean(text):
    text = str(text)
    text = text.lower()
    return text

df["clean_text"] = df["Essay"].apply(basic_clean)

In [ ]:
df["clean_text"]

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', text)

df["tokens"] = df["clean_text"].apply(tokenize)

In [ ]:
df["unique_words"] = df["tokens"].apply(lambda x: len(set(x)))

df["ttr"] = df["unique_words"] / df["length"]

df["avg_word_len"] = df["tokens"].apply(
    lambda x: np.mean([len(w) for w in x]) if len(x) > 0 else 0
)

In [ ]:
def split_sentences(text):
    return re.split(r'[.!?]+', text)

df["sentences"] = df["Essay"].apply(split_sentences)

df["sentence_count"] = df["sentences"].apply(lambda x: len([s for s in x if s.strip() != ""]))

df["avg_sentence_len"] = df.apply(
    lambda row: row["length"] / row["sentence_count"]
    if row["sentence_count"] > 0 else 0,
    axis=1
)

df["sentence_len_var"] = df["sentences"].apply(
    lambda sents: np.var([len(s.split()) for s in sents if s.strip() != ""])
    if len(sents) > 0 else 0
)

In [ ]:
df["long_word_ratio"] = df["tokens"].apply(
    lambda x: np.mean([len(w) > 6 for w in x]) if len(x) > 0 else 0
)

df["short_word_ratio"] = df["tokens"].apply(
    lambda x: np.mean([len(w) <= 3 for w in x]) if len(x) > 0 else 0
)

In [ ]:
def count_punctuation(text):
    return len(re.findall(r'[,.!?;:]', text))

df["punct_count"] = df["Essay"].apply(count_punctuation)

df["punct_density"] = df["punct_count"] / df["length"]

In [ ]:
features = [
    "length",
    "unique_words",
    "ttr",
    "avg_word_len",
    "sentence_count",
    "avg_sentence_len",
    "sentence_len_var",
    "long_word_ratio",
    "short_word_ratio",
    "punct_density",
]

df[features].head()

In [ ]:
corr = df[features + ["Overall_Clean"]].corr()

corr["Overall_Clean"].sort_values(ascending=False)

In [ ]:
len(df)

In [ ]:
# df.to_csv("../data/processed/task2_cleaned_final.csv", index=False)

# print("\nSaved: task2_cleaned_final.csv")